In [ ]:
import pandas as pd
import numpy as np
import math as ma

In [ ]:
df = pd.read_csv("./data/rappelconso0.csv", sep=";")
df = df.rename({"?R?f?rence Fiche": "R?ference Fiche"}, axis=1)

In [ ]:
df.columns.to_list()

In [ ]:
df["Cat?gorie de produit"].unique()

In [ ]:
df[pd.isna(df["Nature juridique du rappel"])]["Cat?gorie de produit"].unique()

In [ ]:
df

In [ ]:
list_util = ["R?ference Fiche", "Date de publication", "Nature juridique du rappel", "Cat?gorie de produit", "Sous-cat?gorie de produit", 
             "Nom de la marque du produit", "Zone g?ographique de vente", "Distributeurs", "Motif du rappel", 
             "Risques encourus par le consommateur", "Modalit?s de compensation", "Date de fin de la proc?dure de rappel"]
df_util = df.copy()
df_util = df_util[list_util]

In [ ]:
df_util

In [ ]:
df_util[["Cat?gorie de produit", "R?ference Fiche"]].groupby(by="Cat?gorie de produit").count()

# 1er PLOT

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# plot cumsum of recalls with hue = product category and time
plt.figure(figsize=(10, 5))
df_util["Date de publication"] = pd.to_datetime(df_util["Date de publication"])
df_util["Date de publication"].hist(bins=100)
plt.xticks(rotation=90)
plt.show()

In [ ]:
def fillna(df, col, value):
    # fillna sur une colonne
    df[col] = df[col].fillna(value)
    return df

def fillnas(df, dictio):
    # plusieurs fillna en m?me temps
    for k, v in dictio.items():
        df = fillna(df, k, v)
    return df

def replace_value_in_col(df, col:str, dictio:dict):
    # remplace les valeurs d'une colonne par d'autres
    df[col] = df[col].replace(dictio)
    return df

def replace_values_in_cols(df, dictio:dict):
    # remplace les valeurs de plusieurs colonnes par d'autres
    for col, values in dictio.items():
        df = replace_value_in_col(df, col, values)
    return df


In [ ]:
df = pd.read_csv("./data/rappelconso0.csv", sep=";")
df = df.rename({"?R?f?rence Fiche": "R?ference Fiche"}, axis=1)

In [ ]:

# list columns which contain NaN values
df = fillnas(df, {"Nature juridique du rappel": "Volontaire (sans arr?t? pr?fectoral)",
                  "Cat?gorie de produit": "Non Cat?gori?",
                  "Sous-cat?gorie de produit": "Non Cat?goris?",
                 #'Zone g?ographique de vente':'France enti?re', 
                  'Distributeurs':'Non Sp?cifi?',
                  'Modalit?s de compensation' : "Non Sp?cifi?",
                  'Date de fin de la proc?dure de rappel' : "Non Sp?cifi?"})


list_na_ok = ["Nom de la marque du produit", 'Noms des mod?les ou r?f?rences',
 'Identification des produits',
 'Conditionnements',
 'Date d?but/Fin de commercialisation',
 'Temp?rature de conservation',
 'Marque de salubrit?',
 'Informations compl?mentaires','Zone g?ographique de vente',
 'Pr?conisations sanitaires',
 'Description compl?mentaire du risque',
 'Conduites ? tenir par le consommateur',
 'Num?ro de contact',
 'Informations compl?mentaires publiques',
 'Liens vers les images',
 'Lien vers la liste des produits',
 'Lien vers la liste des distributeurs'
 ]
[i for i in df.columns[pd.isna(df).any()].tolist() if i not in list_na_ok] 

In [ ]:
df["Distributeurs"] = df["Distributeurs"].apply(lambda x: x.lower() if type(x) == str else x)
df["Distributeurs"] = df["Distributeurs"].apply(lambda x: x.replace("\r\n", ",") if type(x) == str else x)

a = df['Distributeurs'].dropna().unique().tolist()
b = [i for i in a if len(i)> 10]
list_distrib = [i.lower() for i in ["AUCHAN", "CARREFOUR", "SYSTEME U", 
                "CASINO", "INTERMARCHE", "LEADER PRICE", 
                "LECLERC", "ATAC", "CANTREL SALAISON SARL",
                "CORA", "LKB INTERNATIONAL", "LIDL", "MONOPRIX", 
                "AMAZON", "CDISCOUNT", "FRANPRIX", "LEADER PRICE", "FNAC",
                "DARTY", "HYPERMARCH?", "MONOPRIX", " U ", "LA RUCHE QUI DIT OUI"]]
# not any but more than one
c = [i for i in b if sum([j in i.lower() for j in list_distrib]) > 1]
d = [i for i in b if sum([j in i.lower() for j in list_distrib]) == 1]
df.loc[df["Distributeurs"].isin([i for i in c]), "Distributeurs"] = "Plusieurs Grands Distributeurs"
# for each distributeur in df that is also in d, replace by the name of the distributeur in list_distrib
for i in d:
    for j in list_distrib:
        if j in i.lower():
            if j == " U ":
                df.loc[df["Distributeurs"] == i, "Distributeurs"] = "SYSTEME U"
            df.loc[df["Distributeurs"] == i, "Distributeurs"] = j
a = df['Distributeurs'].dropna().unique().tolist()
b = a
c = [i for i in b if sum([j in i.lower() for j in list_distrib]) > 1]
d = [i for i in b if sum([j in i.lower() for j in list_distrib]) == 1]
len(c)

In [ ]:
[i for i in df["Distributeurs"] if "Carrefour" in i]

In [ ]:
(len(set([i for i in b if ((i not in c) and (i not in d))])), set([i for i in b if ((i not in c) and (i not in d))]))

In [ ]:
(len(set(d)), set(d))

In [ ]:
df[df["Distributeurs"] == "LOU YETU\r\nAutres"]